## SQL 분석 및 피처 엔지니어링

### 목적
`02_eda.ipynb`에서 확인한 시차 분석 결과를 반영하여    
모델링에 사용할 master_table을 생성한다.

- kamis_egg_retail + ecos_ppi JOIN
- LAG 변수 생성 (feed_lag1~3, elec_lag1~3)
- regime 더미 컬럼 추가
- master_table DB 적재 및 CSV 저장

> 📎 LAG 1~3 채택 근거: `02_eda.ipynb` 시차 분석에서 4개월 이후 설명력 개선이 미미함을 확인

In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../egg_analysis.db")

print(pd.read_sql("SELECT * FROM kamis_egg_retail LIMIT 5", conn))
print(pd.read_sql("SELECT * FROM ecos_ppi LIMIT 5", conn))

            year_month  egg_price
0  2016-01-01 00:00:00       5493
1  2016-02-01 00:00:00       5473
2  2016-03-01 00:00:00       5260
3  2016-04-01 00:00:00       5259
4  2016-05-01 00:00:00       5216
            year_month  egg_ppi  electricity_ppi  feed_ppi  egg_cpi
0  2016-01-01 00:00:00    92.17            98.81     95.69   92.594
1  2016-02-01 00:00:00    85.08            98.81     95.69   91.098
2  2016-03-01 00:00:00    83.52            98.81     94.82   87.269
3  2016-04-01 00:00:00    89.81            98.81     94.82   86.773
4  2016-05-01 00:00:00    85.53            98.81     94.82   84.645


In [3]:
query_join = """
SELECT
    a.year_month,
    a.egg_price,
    b.feed_ppi,
    b.electricity_ppi,
    b.egg_ppi,
    b.egg_cpi
FROM kamis_egg_retail a
JOIN ecos_ppi b ON a.year_month = b.year_month
ORDER BY a.year_month
"""

df_join = pd.read_sql(query_join, conn)

print("JOIN 결과 행 수:", df_join.shape)  
print(df_join.head())
print(df_join.isnull().sum())

JOIN 결과 행 수: (120, 6)
            year_month  egg_price  feed_ppi  electricity_ppi  egg_ppi  egg_cpi
0  2016-01-01 00:00:00       5493     95.69            98.81    92.17   92.594
1  2016-02-01 00:00:00       5473     95.69            98.81    85.08   91.098
2  2016-03-01 00:00:00       5260     94.82            98.81    83.52   87.269
3  2016-04-01 00:00:00       5259     94.82            98.81    89.81   86.773
4  2016-05-01 00:00:00       5216     94.82            98.81    85.53   84.645
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
dtype: int64


In [4]:
query_lag = """
SELECT
    year_month,
    egg_price,
    feed_ppi,
    electricity_ppi,
    egg_ppi,
    egg_cpi,
    LAG(feed_ppi, 1) OVER (ORDER BY year_month) AS feed_lag1,
    LAG(feed_ppi, 2) OVER (ORDER BY year_month) AS feed_lag2,
    LAG(feed_ppi, 3) OVER (ORDER BY year_month) AS feed_lag3,
    LAG(electricity_ppi, 1) OVER (ORDER BY year_month) AS elec_lag1,
    LAG(electricity_ppi, 2) OVER (ORDER BY year_month) AS elec_lag2,
    LAG(electricity_ppi, 3) OVER (ORDER BY year_month) AS elec_lag3,
    CASE WHEN SUBSTR(year_month, 1, 4) >= '2021' THEN 1 ELSE 0 END AS regime
FROM (
    SELECT
        a.year_month,
        a.egg_price,
        b.feed_ppi,
        b.electricity_ppi,
        b.egg_ppi,
        b.egg_cpi
    FROM kamis_egg_retail a
    JOIN ecos_ppi b ON a.year_month = b.year_month
    ORDER BY a.year_month
)
"""

df_master = pd.read_sql(query_lag, conn)
print("LAG 변수 추가 후:", df_master.shape) 
print(df_master.head())
print(df_master.isnull().sum())

LAG 변수 추가 후: (120, 13)
            year_month  egg_price  feed_ppi  electricity_ppi  egg_ppi  \
0  2016-01-01 00:00:00       5493     95.69            98.81    92.17   
1  2016-02-01 00:00:00       5473     95.69            98.81    85.08   
2  2016-03-01 00:00:00       5260     94.82            98.81    83.52   
3  2016-04-01 00:00:00       5259     94.82            98.81    89.81   
4  2016-05-01 00:00:00       5216     94.82            98.81    85.53   

   egg_cpi  feed_lag1  feed_lag2  feed_lag3  elec_lag1  elec_lag2  elec_lag3  \
0   92.594        NaN        NaN        NaN        NaN        NaN        NaN   
1   91.098      95.69        NaN        NaN      98.81        NaN        NaN   
2   87.269      95.69      95.69        NaN      98.81      98.81        NaN   
3   86.773      94.82      95.69      95.69      98.81      98.81      98.81   
4   84.645      94.82      94.82      95.69      98.81      98.81      98.81   

   regime  
0       0  
1       0  
2       0  
3       0

In [5]:
df_master = df_master.dropna().reset_index(drop=True)

print("결측치 제거 후:", len(df_master), "행")  
print(df_master.isnull().sum())  

결측치 제거 후: 117 행
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
feed_lag1          0
feed_lag2          0
feed_lag3          0
elec_lag1          0
elec_lag2          0
elec_lag3          0
regime             0
dtype: int64


In [6]:
df_master.to_sql("master_table", conn, if_exists="replace", index=False)

# 검증 
df_master_db = pd.read_sql("SELECT * FROM master_table", conn)

print("적재 전 행 수:", len(df_master))
print("적재 후 행 수:", len(df_master_db))
print("일치 여부:", len(df_master) == len(df_master_db))
print(df_master_db.isnull().sum())

적재 전 행 수: 117
적재 후 행 수: 117
일치 여부: True
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
feed_lag1          0
feed_lag2          0
feed_lag3          0
elec_lag1          0
elec_lag2          0
elec_lag3          0
regime             0
dtype: int64


### master_table 구조 설명 
| 컬럼 | 설명 | 모델링 역할 |
|------|------|------------|
| egg_price | 달걀 소매가격 (원/30개) | 종속변수 Y |
| feed_ppi | 사료 생산자물가지수 | 핵심 원가 피처 |
| electricity_ppi | 전력 생산자물가지수 | 보조 원가 피처 |
| feed_lag1~3 | 사료비 1~3개월 시차 | 시차 전이 포착 |
| elec_lag1~3 | 전력 1~3개월 시차 | 시차 전이 포착 |
| regime | 2021년 이후 고가 레짐 더미 | 구조적 가격 바닥 상향 반영 |
| egg_ppi, egg_cpi | 달걀 물가지수 | 참고용 (모델 피처 미사용) |        

In [7]:
df_master.to_csv("../data/processed/master_table.csv", index=False)
print("마스터 테이블 CSV 저장 완료")

마스터 테이블 CSV 저장 완료
